Imports / Configurações

In [9]:
import pandas as pd
import os
import sys
from datasets import load_dataset

MIN_WORDS = 80
MAX_WORDS = 120

## Load

In [3]:
def is_valid_text(text):
    """Verifica se o texto é uma string válida e tem o tamanho certo."""
    if not isinstance(text, str):
        return False
    word_count = len(text.split())
    return MIN_WORDS <= word_count <= MAX_WORDS


In [4]:
print("A descarregar o dataset.")
dataset = load_dataset("dmitva/human_ai_generated_text", split="train")

human_count = 0
dataset_final = []

print("A filtrar textos entre 80 e 120 palavras.")
for row in dataset:
    texto_humano = str(row.get('human_text', ""))
    
    if is_valid_text(texto_humano):
        dataset_final.append({'text': texto_humano, 'label': 'Human'})
        human_count += 1

print(f"Textos 'Human' recolhidos: {human_count}")

df = pd.DataFrame(dataset_final)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "dataset_human_filtrado.csv")
df.to_csv(output_path, index=False)

A descarregar o dataset.
A filtrar textos entre 80 e 120 palavras.
Textos 'Human' recolhidos: 6180


## Concatenar + Shuffle

In [5]:
df_human = df.sample(n=120, random_state=2026)

df_openai = pd.read_csv("../data/dataset_openai.csv")
df_anthropic = pd.read_csv("../data/dataset_anthropic.csv")
df_google = pd.read_csv("../data/dataset_google.csv")
df_meta = pd.read_csv("../data/dataset_meta.csv")

In [6]:
lista_dfs = [df_human, df_openai, df_anthropic, df_google, df_meta]
df_completo = pd.concat(lista_dfs, ignore_index=True)
df_completo = df_completo.sample(frac=1, random_state=2026).reset_index(drop=True)

In [7]:
df_completo.head(10)

,text,label
0,Software engineering is the systematic applica...,Meta
1,"Cell biology focuses on the study of cells, th...",Meta
2,Cells are the basic structural and functional ...,OpenAI
3,Acoustics is the scientific discipline dedicat...,Google
4,Condensed matter physics is the largest branch...,Meta
5,I think is important because de Churchill have...,Human
6,Electromagnetism studies electric and magnetic...,OpenAI
7,Geophysics is a branch of physics concerned wi...,Meta
8,"Mycology is the scientific study of fungi, a d...",Meta
9,Civil engineering encompasses the design and c...,Anthropic


### Escrita

In [10]:
df_completo.to_csv("../data/dataset_completo.csv")